# 07 — Sensitivity

OAT tornado and normalized $S_x^{V_{\mathrm{OUT}}}=(x/V)\,\partial V/\partial x$. Use this to decide whether 0.1 % feedback resistors are worth it.

## Setup — editable BOM and run knobs

This cell is shared by every notebook. It puts `P5V_Regulator/` on `sys.path`
and builds **one** `CircuitParams` object. Change a resistor, capacitor, or
tolerance here and re-run: analytical, SPICE, WC, and MC all follow this object.

**Plot callouts:** each figure cell has an `ann = [dict(...), ...]` list.
Edit `text`, `xy` (data coords of the arrow tip), and `offset=(dx, dy)` in
**points** to nudge a label. Use `xytext=(x, y)` instead of `offset` for data
coordinates. `arrow=False` for a plain label. `ann = []` hides all callouts.

Tolerances are **fractions** (0.01 = 1 %). Figures go to `results/figures/`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_project_root() -> Path:
    start = Path.cwd().resolve()
    named = Path("Sections/Electronics/Analog_and_Digital/LT3010_5V_Supply")
    candidates = [start, *start.parents, start / named]
    if start.name == "notebooks":
        candidates.insert(0, start.parent)
    for cand in candidates:
        if (cand / "p5v_design" / "__init__.py").is_file():
            return cand
    return start

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from p5v_design.params import CircuitParams, ToleranceSpec, format_eng, LdoParams
from p5v_design import analysis, schematic, plots, stability
from p5v_design import plots_wc_mc as pwc
from p5v_design.ldo_model import dropout_v, ignd_a, tj_c, psrr_db, vadj_typical_vs_temp
from dataclasses import replace

# --- EDITABLE BOM ---
# Datasheet TA01, "5V Supply with Shutdown" (LT3010/LT3010-5 Rev. J page 1).
# Fixed 5 V part: SENSE tied to OUT (R1 = 0). No external divider (R2 open).
# Capacitors are the 1 µF parts drawn on that figure. Load is the labeled 50 mA.
# VIN is printed as 5.4 V to 80 V; 12 V is the single-point bias inside that range.
R1 = 0.0             # SENSE shorted to OUT
R2 = 1.0e12          # open — no bottom resistor
CFF = 0.0            # no feedforward capacitor on the fixed part
COUT = 1.0e-6
CIN = 1.0e-6
ESR_OUT = 10e-3      # ceramic assumption, not printed on TA01
ESR_IN = 10e-3
RLOAD = 100.0        # 5 V / 50 mA
R_TOL = 0.01         # 1 %
C_TOL = 0.10         # 10 %
VIN_NOM = 12.0
VIN_MIN = 5.4
VIN_MAX = 80.0
ILOAD_OP = 50e-3
N_MC = 2000
N_SPICE_MC = 40
N_LTSPICE_MC = 200
SKIP_SPICE_WC_MC = False
RUN_LTSPICE_BATCH = False
SPICE_WORKERS = None
LTSPICE_WORKERS = None

p = CircuitParams(
    r1=R1, r2=R2, cff=CFF, cout=COUT, cin=CIN,
    esr_out=ESR_OUT, esr_in=ESR_IN, r_load=RLOAD,
    r_tol=ToleranceSpec(R_TOL), c_tol=ToleranceSpec(C_TOL),
)
p = replace(p, op=replace(p.op, vin_nom=VIN_NOM, vin_min=VIN_MIN, vin_max=VIN_MAX,
                           iload_op=ILOAD_OP, r_load=RLOAD))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("gain =", p.gain, "  Vout nom =", analysis.vout_dc(p))
print("SPICE_WORKERS =", SPICE_WORKERS, "  N_MC =", N_MC)


gain = 1.0   Vout nom = 5.0
SPICE_WORKERS = None   N_MC = 2000


In [2]:
from p5v_design.sensitivity import full_sensitivity
sens = full_sensitivity(p)
display(sens.oat)
display(sens.normalized)
ann = []
torn = dict(zip(sens.oat["name"], sens.oat["half_span_vout_mV"]))
plots.plot_tornado(torn, xlabel="half-span (mV)", title="Vout sensitivity",
                   annotations=ann, path=FIG/"07_tornado.png")
plt.show()
# What-if 0.1 % resistors
p01 = p.with_values(r_tol=ToleranceSpec(0.001))
s01 = full_sensitivity(p01)
print("RSS 1 %", np.sqrt(np.sum(np.square(sens.oat.half_span_vout_mV))))
print("RSS 0.1 %", np.sqrt(np.sum(np.square(s01.oat.half_span_vout_mV))))
plots.plot_oat_share(sens.oat, path=FIG/"07_oat_share.png")
plt.show()
from p5v_design.worst_case import oat_contributions
a = oat_contributions(p)["vout"]
b = oat_contributions(p01)["vout"]
plots.plot_tornado_compare(a, b, path=FIG/"07_tornado_1pct_vs_0p1.png")
plt.show()
display(sens.normalized)
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.barh(sens.normalized["name"], sens.normalized["S"], color="C2", edgecolor="k", lw=0.4)
ax.set_xlabel(r"$S_x^{V_{OUT}}=(x/V)\,\partial V/\partial x$")
ax.set_title("Normalized DC sensitivity")
ax.grid(True, axis="x", alpha=0.3)
plots.savefig(fig, FIG/"07_normalized_S.png")
plt.show()
# Other-metric tornados (HV_monitor 07)
from p5v_design.worst_case import oat_contributions as oat_c
oat_all = oat_c(p, metrics=("vout", "headroom", "p_diss", "efficiency"))
for met, xlab, fn in (
    ("headroom", "half-span (V)", "07_tornado_headroom.png"),
    ("p_diss", "half-span (W)", "07_tornado_pdiss.png"),
    ("efficiency", "half-span", "07_tornado_eta.png"),
):
    plots.plot_tornado(oat_all[met], xlabel=xlab, title=f"OAT tornado — {met}", path=FIG/fn)
    plt.show()
s_ot = full_sensitivity(p, over_temp=True)
plots.plot_tornado_compare(
    dict(zip(sens.oat["name"], sens.oat["half_span_vout"])),
    dict(zip(s_ot.oat["name"], s_ot.oat["half_span_vout"])),
    label_a="25 °C", label_b="over temp",
    title="Vout sensitivity  25 °C vs over-temp envelope",
    path=FIG/"07_tornado_25_vs_ot.png",
)
plt.show()
plots.plot_psrr_vs_cout(p, annotations=[
    dict(text="BOM 10 µF", xy=(10.0, 75), offset=(12, -14), arrow=True),
], path=FIG/"07_psrr_vs_cout.png")
plt.show()

,name,half_span_vout_mV,half_span_vout,pct_of_rss
0,Vadj,75.0,0.075,100.0
1,Iadj,0.0,0.000,0.0
2,R1,0.0,0.000,0.0
3,R2,0.0,0.000,0.0


,name,dVout_dx,S
0,r1,0.0,0.0
1,r2,0.0,0.0
2,vadj,1.0,1.0
3,iadj,0.0,0.0


RSS 1 % 75.00000000000017
RSS 0.1 % 75.00000000000017


cell:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,name,dVout_dx,S
0,r1,0.0,0.0
1,r2,0.0,0.0
2,vadj,1.0,1.0
3,iadj,0.0,0.0


cell:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
